In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import os
import re
import joblib
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBRegressor

In [2]:
# ============================================================
# 2. DOWNLOAD AND LOAD DATASET
# ============================================================

path = kagglehub.dataset_download(
    "asaniczka/amazon-products-dataset-2023-1-4m-products"
)

print("Dataset Path:", path)

print("\nFiles in dataset:")
print(os.listdir(path))

products_file = os.path.join(path, "amazon_products.csv")
categories_file = os.path.join(path, "amazon_categories.csv")

df = pd.read_csv(products_file)
cat_df = pd.read_csv(categories_file)

print("Products Shape:", df.shape)
print("Categories Shape:", cat_df.shape)

print("\nProduct Columns:")
print(df.columns.tolist())

print("\nCategory Columns:")
print(cat_df.columns.tolist())

df.head()

Dataset Path: C:\Users\Shivani Agarwal\.cache\kagglehub\datasets\asaniczka\amazon-products-dataset-2023-1-4m-products\versions\17

Files in dataset:
['amazon_categories.csv', 'amazon_products.csv']
Products Shape: (1426337, 11)
Categories Shape: (248, 2)

Product Columns:
['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']

Category Columns:
['id', 'category_name']


,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",https://m.media-amazon.com/images/I/815dLQKYIY...,https://www.amazon.com/dp/B014TMV5YE,4.5,0,139.99,0.00,104,False,2000
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,https://m.media-amazon.com/images/I/81bQlm7vf6...,https://www.amazon.com/dp/B07GDLCQXV,4.5,0,169.99,209.99,104,False,1000
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,https://m.media-amazon.com/images/I/71EA35zvJB...,https://www.amazon.com/dp/B07XSCCZYG,4.6,0,365.49,429.99,104,False,300
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,https://m.media-amazon.com/images/I/91k6NYLQyI...,https://www.amazon.com/dp/B08MVFKGJM,4.6,0,291.59,354.37,104,False,400
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,https://m.media-amazon.com/images/I/61NJoaZcP9...,https://www.amazon.com/dp/B01DJLKZBA,4.5,0,174.99,309.99,104,False,400


In [3]:
# ============================================================
# 3. MERGE CATEGORY NAME
# ============================================================

cat_df.columns = cat_df.columns.str.strip()

df = df.merge(
    cat_df,
    left_on="category_id",
    right_on="id",
    how="left"
)

print("Shape after category merge:", df.shape)

df[["title", "category_id", "category_name"]].head()

Shape after category merge: (1426337, 13)


,title,category_id,category_name
0,"Sion Softside Expandable Roller Luggage, Black...",104,Suitcases
1,Luggage Sets Expandable PC+ABS Durable Suitcas...,104,Suitcases
2,Platinum Elite Softside Expandable Checked Lug...,104,Suitcases
3,Freeform Hardside Expandable with Double Spinn...,104,Suitcases
4,Winfield 2 Hardside Expandable Luggage with Sp...,104,Suitcases


In [4]:
# ============================================================
# 4. CLEAN DATA
# ============================================================

def clean_price(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    x = x.replace("$", "")
    x = x.replace(",", "")
    x = x.strip()
    
    match = re.findall(r"\d+\.?\d*", x)
    
    if match:
        return float(match[0])
    
    return np.nan


df["price_clean"] = df["price"].apply(clean_price)
df["listPrice_clean"] = df["listPrice"].apply(clean_price)

df["stars"] = pd.to_numeric(df["stars"], errors="coerce").fillna(0)
df["reviews"] = pd.to_numeric(df["reviews"], errors="coerce").fillna(0)
df["boughtInLastMonth"] = pd.to_numeric(
    df["boughtInLastMonth"], 
    errors="coerce"
).fillna(0)

df["isBestSeller"] = (
    df["isBestSeller"]
    .astype(str)
    .str.lower()
    .map({"true": 1, "false": 0})
    .fillna(0)
)

df = df.dropna(subset=["price_clean"])
df = df[df["price_clean"] > 0]

upper_limit = df["price_clean"].quantile(0.99)
df = df[df["price_clean"] <= upper_limit]

print("Shape after cleaning:", df.shape)

df[
    [
        "asin",
        "title",
        "price",
        "price_clean",
        "listPrice",
        "listPrice_clean",
        "category_name",
        "stars",
        "reviews",
        "isBestSeller",
        "boughtInLastMonth"
    ]
].head()

Shape after cleaning: (1379629, 15)


,asin,title,price,price_clean,listPrice,listPrice_clean,category_name,stars,reviews,isBestSeller,boughtInLastMonth
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",139.99,139.99,0.00,0.00,Suitcases,4.5,0,0,2000
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,169.99,169.99,209.99,209.99,Suitcases,4.5,0,0,1000
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,365.49,365.49,429.99,429.99,Suitcases,4.6,0,0,300
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,291.59,291.59,354.37,354.37,Suitcases,4.6,0,0,400
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,174.99,174.99,309.99,309.99,Suitcases,4.5,0,0,400


In [5]:
# ============================================================
# 5. VISUAL ATTRIBUTE EXTRACTION FROM TITLE
# ============================================================
# Since we are not downloading images here, we extract product attributes
# from title text. Later this same logic can be applied to OCR text.

def extract_wattage(text):
    text = str(text).lower()
    match = re.search(r"(\d+\.?\d*)\s*(w|watt|watts)\b", text)
    return float(match.group(1)) if match else 0


def extract_voltage(text):
    text = str(text).lower()
    match = re.search(r"(\d+\.?\d*)\s*(v|volt|volts)\b", text)
    return float(match.group(1)) if match else 0


def extract_weight_kg(text):
    text = str(text).lower()
    
    kg_match = re.search(r"(\d+\.?\d*)\s*(kg|kilogram|kilograms)\b", text)
    gram_match = re.search(r"(\d+\.?\d*)\s*(g|gram|grams)\b", text)
    lb_match = re.search(r"(\d+\.?\d*)\s*(lb|lbs|pound|pounds)\b", text)
    
    if kg_match:
        return float(kg_match.group(1))
    elif gram_match:
        return float(gram_match.group(1)) / 1000
    elif lb_match:
        return float(lb_match.group(1)) * 0.453592
    
    return 0


def extract_volume_litre(text):
    text = str(text).lower()
    
    litre_match = re.search(r"(\d+\.?\d*)\s*(l|litre|liter|liters|litres)\b", text)
    ml_match = re.search(r"(\d+\.?\d*)\s*(ml|milliliter|millilitre)\b", text)
    oz_match = re.search(r"(\d+\.?\d*)\s*(fl oz|oz)\b", text)
    
    if litre_match:
        return float(litre_match.group(1))
    elif ml_match:
        return float(ml_match.group(1)) / 1000
    elif oz_match:
        return float(oz_match.group(1)) * 0.0295735
    
    return 0


def extract_quantity(text):
    text = str(text).lower()
    
    pack_match = re.search(r"pack\s*of\s*(\d+)", text)
    count_match = re.search(r"(\d+)\s*(pcs|pieces|count|counts|ct|units)\b", text)
    
    if pack_match:
        return int(pack_match.group(1))
    elif count_match:
        return int(count_match.group(1))
    
    return 1


df["visual_wattage"] = df["title"].apply(extract_wattage)
df["visual_voltage"] = df["title"].apply(extract_voltage)
df["visual_weight_kg"] = df["title"].apply(extract_weight_kg)
df["visual_volume_litre"] = df["title"].apply(extract_volume_litre)
df["visual_quantity"] = df["title"].apply(extract_quantity)

df[
    [
        "title",
        "visual_wattage",
        "visual_voltage",
        "visual_weight_kg",
        "visual_volume_litre",
        "visual_quantity"
    ]
].head(20)

,title,visual_wattage,visual_voltage,visual_weight_kg,visual_volume_litre,visual_quantity
0,"Sion Softside Expandable Roller Luggage, Black...",0.0,0.0,0.0,0.0,1
1,Luggage Sets Expandable PC+ABS Durable Suitcas...,0.0,0.0,0.0,0.0,1
2,Platinum Elite Softside Expandable Checked Lug...,0.0,0.0,0.0,0.0,1
3,Freeform Hardside Expandable with Double Spinn...,0.0,0.0,0.0,0.0,1
4,Winfield 2 Hardside Expandable Luggage with Sp...,0.0,0.0,0.0,0.0,1
5,Maxlite 5 Softside Expandable Luggage with 4 S...,0.0,0.0,0.0,0.0,1
6,"Hard Shell Carry on Luggage Airline Approved, ...",0.0,0.0,0.0,0.0,1
7,"Maxporter II 30"" Hardside Spinner Trunk Luggag...",0.0,0.0,0.0,0.0,1
8,Omni 2 Hardside Expandable Luggage with Spinne...,0.0,0.0,0.0,0.0,1
9,Luggage Sets Expandable Lightweight Suitcases ...,0.0,0.0,0.0,0.0,4


In [6]:
# ============================================================
# 6. CREATE TRAINING DATA
# ============================================================

model_df = pd.DataFrame()

model_df["title"] = df["title"].fillna("")
model_df["category"] = df["category_name"].fillna("unknown")

model_df["stars"] = df["stars"]
model_df["reviews"] = df["reviews"]
model_df["listPrice"] = df["listPrice_clean"].fillna(0)
model_df["isBestSeller"] = df["isBestSeller"]
model_df["boughtInLastMonth"] = df["boughtInLastMonth"]

model_df["visual_wattage"] = df["visual_wattage"]
model_df["visual_voltage"] = df["visual_voltage"]
model_df["visual_weight_kg"] = df["visual_weight_kg"]
model_df["visual_volume_litre"] = df["visual_volume_litre"]
model_df["visual_quantity"] = df["visual_quantity"]

model_df["price"] = df["price_clean"]

model_df.head()

,title,category,stars,reviews,listPrice,isBestSeller,boughtInLastMonth,visual_wattage,visual_voltage,visual_weight_kg,visual_volume_litre,visual_quantity,price
0,"Sion Softside Expandable Roller Luggage, Black...",Suitcases,4.5,0,0.00,0,2000,0.0,0.0,0.0,0.0,1,139.99
1,Luggage Sets Expandable PC+ABS Durable Suitcas...,Suitcases,4.5,0,209.99,0,1000,0.0,0.0,0.0,0.0,1,169.99
2,Platinum Elite Softside Expandable Checked Lug...,Suitcases,4.6,0,429.99,0,300,0.0,0.0,0.0,0.0,1,365.49
3,Freeform Hardside Expandable with Double Spinn...,Suitcases,4.6,0,354.37,0,400,0.0,0.0,0.0,0.0,1,291.59
4,Winfield 2 Hardside Expandable Luggage with Sp...,Suitcases,4.5,0,309.99,0,400,0.0,0.0,0.0,0.0,1,174.99


In [12]:
# ============================================================
# 7. SAMPLE DATA FOR FASTER TRAINING
# ============================================================

SAMPLE_SIZE = 100000

if len(model_df) > SAMPLE_SIZE:
    model_df = model_df.sample(SAMPLE_SIZE, random_state=42)

print("Model Dataset Shape:", model_df.shape)

Model Dataset Shape: (100000, 13)


In [13]:
# ============================================================
# 8. TRAIN TEST SPLIT
# ============================================================

X = model_df.drop(columns=["price"])
y = model_df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (80000, 12)
Test Shape: (20000, 12)


In [14]:
# ============================================================
# 9. FEATURE ENGINEERING + MODEL PIPELINE
# ============================================================

text_col = "title"
cat_cols = ["category"]

num_cols = [
    "stars",
    "reviews",
    "listPrice",
    "isBestSeller",
    "boughtInLastMonth",
    "visual_wattage",
    "visual_voltage",
    "visual_weight_kg",
    "visual_volume_litre",
    "visual_quantity"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "title_tfidf",
            TfidfVectorizer(
                max_features=15000,
                stop_words="english",
                ngram_range=(1, 2)
            ),
            text_col
        ),
        (
            "category_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                max_categories=100
            ),
            cat_cols
        ),
        (
            "numeric_features",
            "passthrough",
            num_cols
        )
    ]
)

model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [15]:
# ============================================================
# 10. MODEL FINE-TUNING USING RANDOMIZED SEARCH
# ============================================================

from sklearn.model_selection import RandomizedSearchCV

# Base pipeline
base_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

# Hyperparameter search space
param_grid = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [4, 6, 8, 10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.1, 0.5, 1],
    "model__reg_lambda": [1, 1.5, 2, 3]
}

fine_tuner = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_grid,
    n_iter=15,                 # Increase to 30 later
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

fine_tuner.fit(X_train, y_train)

print("Fine-tuning complete.")
print("Best Parameters:")
print(fine_tuner.best_params_)

print("Best MAE Score:")
print(abs(fine_tuner.best_score_))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fine-tuning complete.
Best Parameters:
{'model__subsample': 0.7, 'model__reg_lambda': 3, 'model__reg_alpha': 1, 'model__n_estimators': 400, 'model__max_depth': 8, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.9}
Best MAE Score:
18.121570106887045


In [22]:
# ============================================================
# SAVE FINE-TUNED MODEL
# ============================================================

os.makedirs("models", exist_ok=True)

joblib.dump(pipeline, "models/fine_tuned_amazon_price_model.pkl")

print("Fine-tuned model saved successfully.")

Fine-tuned model saved successfully.


In [23]:
# ============================================================
# 10. TRAIN MODEL
# ============================================================
pipeline = fine_tuner.best_estimator_
print("Training complete.")

Training complete.


In [24]:
# ============================================================
# 11. EVALUATE FINE-TUNED MODEL
# ============================================================

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Fine-Tuned Model Performance")
print("----------------------------")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

Fine-Tuned Model Performance
----------------------------
MAE: 17.96
RMSE: 34.97
R2 Score: 0.4741


In [25]:
# ============================================================
# 12. PRICE BUCKET CLASSIFICATION METRICS
# ============================================================

def price_bucket(price):
    if price <= 25:
        return "low"
    elif price <= 100:
        return "medium"
    elif price <= 500:
        return "high"
    else:
        return "premium"


y_test_bucket = y_test.apply(price_bucket)
y_pred_bucket = pd.Series(y_pred).apply(price_bucket)

accuracy = accuracy_score(y_test_bucket, y_pred_bucket)
precision = precision_score(
    y_test_bucket,
    y_pred_bucket,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    y_test_bucket,
    y_pred_bucket,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    y_test_bucket,
    y_pred_bucket,
    average="weighted",
    zero_division=0
)

print("Price Bucket Classification Metrics")
print("-----------------------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

print("\nClassification Report:")
print(classification_report(y_test_bucket, y_pred_bucket, zero_division=0))

Price Bucket Classification Metrics
-----------------------------------
Accuracy: 0.6286
Precision: 0.7217
Recall: 0.6286
F1 Score: 0.6398

Classification Report:
              precision    recall  f1-score   support

        high       0.72      0.47      0.56      1358
         low       0.86      0.56      0.68     12337
      medium       0.45      0.79      0.57      6305

    accuracy                           0.63     20000
   macro avg       0.68      0.61      0.61     20000
weighted avg       0.72      0.63      0.64     20000



In [ ]:
# ============================================================
# 13. ACTUAL VS PREDICTED RESULTS
# ============================================================

results = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": y_pred
})

results["absolute_error"] = abs(
    results["actual_price"] - results["predicted_price"]
)

results.head(20)

In [21]:
# ============================================================
# 14. GRAPH 1: ACTUAL VS PREDICTED PRICE
# ============================================================

plt.figure(figsize=(8, 6))
plt.scatter(
    results["actual_price"],
    results["predicted_price"],
    alpha=0.4
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Product Prices")

min_price = min(results["actual_price"].min(), results["predicted_price"].min())
max_price = max(results["actual_price"].max(), results["predicted_price"].max())

plt.plot([min_price, max_price], [min_price, max_price])

plt.show()

NameError: name 'results' is not defined

<Figure size 800x600 with 0 Axes>

In [ ]:
# ============================================================
# 15. GRAPH 2: ERROR DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 6))
plt.hist(results["absolute_error"], bins=50)

plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")

plt.show()

In [ ]:
# ============================================================
# 16. GRAPH 3: TOP 20 ACTUAL VS PREDICTED
# ============================================================

sample_results = results.head(20).copy()
sample_results["index"] = range(1, len(sample_results) + 1)

plt.figure(figsize=(10, 6))

plt.plot(
    sample_results["index"],
    sample_results["actual_price"],
    marker="o",
    label="Actual Price"
)

plt.plot(
    sample_results["index"],
    sample_results["predicted_price"],
    marker="o",
    label="Predicted Price"
)

plt.xlabel("Sample Product")
plt.ylabel("Price")
plt.title("Actual vs Predicted Price for 20 Products")
plt.legend()

plt.show()

In [ ]:
# ============================================================
# 17. SAVE MODEL
# ============================================================

os.makedirs("models", exist_ok=True)

joblib.dump(pipeline, "models/amazon_visual_smart_price_model.pkl")

print("Model saved at: models/amazon_visual_smart_price_model.pkl")

In [ ]:
# ============================================================
# 18. TEST SINGLE PRODUCT PREDICTION
# ============================================================

loaded_model = joblib.load("models/amazon_visual_smart_price_model.pkl")

sample = pd.DataFrame([{
    "title": "Samsung Galaxy Smartphone 128GB 8GB RAM",
    "category": "Cell Phones",
    "stars": 4.5,
    "reviews": 2500,
    "listPrice": 799.99,
    "isBestSeller": 0,
    "boughtInLastMonth": 1000,
    "visual_wattage": 0,
    "visual_voltage": 0,
    "visual_weight_kg": 0,
    "visual_volume_litre": 0,
    "visual_quantity": 1
}])

prediction = loaded_model.predict(sample)[0]

print("Predicted Price: $", round(float(prediction), 2))

In [ ]:
# ============================================================
# 19. FINAL COMBINED OUTPUT
# ============================================================

def build_entity_prediction(row):
    entities = []
    
    if row["visual_weight_kg"] > 0:
        entities.append(f'{round(row["visual_weight_kg"] * 1000)} gram')
    
    if row["visual_voltage"] > 0:
        entities.append(f'{round(row["visual_voltage"])} volt')
    
    if row["visual_wattage"] > 0:
        entities.append(f'{round(row["visual_wattage"])} watt')
    
    if row["visual_volume_litre"] > 0:
        entities.append(f'{round(row["visual_volume_litre"], 2)} litre')
    
    if row["visual_quantity"] > 1:
        entities.append(f'{round(row["visual_quantity"])} count')
    
    if len(entities) == 0:
        return "not_found"
    
    return ", ".join(entities)


final_output = X_test.copy()
final_output["actual_price"] = y_test.values
final_output["predicted_price"] = y_pred
final_output["entity_prediction"] = final_output.apply(
    build_entity_prediction,
    axis=1
)

final_output["final_prediction"] = (
    final_output["entity_prediction"]
    + " | Predicted Price: $"
    + final_output["predicted_price"].round(2).astype(str)
)

final_output = final_output.reset_index(drop=True)
final_output.insert(0, "index", final_output.index)

final_output[
    [
        "index",
        "entity_prediction",
        "predicted_price",
        "final_prediction"
    ]
].head(20)

In [ ]:
# ============================================================
# 20. SAVE FINAL OUTPUT
# ============================================================

os.makedirs("outputs", exist_ok=True)

final_output[
    [
        "index",
        "entity_prediction",
        "predicted_price",
        "final_prediction"
    ]
].to_csv("outputs/final_multimodal_predictions.csv", index=False)

print("Saved at: outputs/final_multimodal_predictions.csv")